# Ajuste e formatação de atributos

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import re

ACCENTS_FROM = "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
ACCENTS_TO   = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"

def remover_acentos(texto: str) -> str:
    return texto.translate(str.maketrans(ACCENTS_FROM, ACCENTS_TO))

def limpar_nome_generico(nome: str) -> str:
    nome = remover_acentos(nome.strip().lower())
    nome = re.sub(r"\(.*?\)", "", nome)
    nome = re.sub(r"[ ,;{}()\n\t=/-]+", "_", nome)
    return nome.strip("_")

def padronizar_texto(df, colunas=None):
    alvo = colunas or [c for c, t in df.dtypes if t == "string"]
    for c in alvo:
        df = df.withColumn(c, F.upper(F.trim(F.translate(F.col(c), ACCENTS_FROM, ACCENTS_TO))))
    return df

def corrigir_mojibake_colunas(df, colunas):
    for c in colunas:
        df = df.withColumn(c, F.decode(F.encode(F.col(c), "ISO-8859-1"), "UTF-8"))
    return df

def numero_br_para_double(df, colunas):
    for c in colunas:
        valor = F.col(c).cast("string")
        valor = F.regexp_replace(valor, r'["\n\r]', "")   # remove aspas soltas e quebras de linha
        valor = F.trim(valor)
        valor = F.regexp_replace(valor, r"\.", "")          # remove separador de milhar
        valor = F.regexp_replace(valor, ",", ".")            # vírgula decimal -> ponto
        df = df.withColumn(c, valor.try_cast(DoubleType()))
    return df

def numero_us_para_double(df, colunas):
    """Formato americano: vírgula = milhar, ponto = decimal."""
    for c in colunas:
        valor = F.col(c).cast("string")
        valor = F.regexp_replace(valor, r'["\n\r]', "")  # aspas soltas e quebras de linha
        valor = F.trim(valor)
        valor = F.regexp_replace(valor, ",", "")           # remove separador de milhar (vírgula)
        df = df.withColumn(c, valor.try_cast(DoubleType()))
    return df

# ler tabelas do Bronze

In [0]:
df_bmp = spark.table("PUC_Sprint_2.anp.bronze_bmp")
df_bar = spark.table("PUC_Sprint_2.anp.bronze_bar")
df_bdep = spark.table("PUC_Sprint_2.anp.bronze_bdep")
df_cambio = spark.table("PUC_Sprint_2.anp.bronze_cambio")
df_brent = spark.table("PUC_Sprint_2.anp.bronze_brent")

ajustes tabelas individuais

In [0]:
print(df_bar.columns)  
print(df_bdep.columns) 
print(df_bmp.columns) 
print(df_brent.columns) 
print(df_cambio.columns) 

In [0]:
nomes_bar = [
    "ano", "campo", "bacia", "estado", "voip_bbl", "vgip_m3",
    "petroleo_acumulado_bbl", "gas_natural_acumulado_m3",
    "fracao_recuperada_petroleo", "situacao",
]
df_bar_silver = df_bar.toDF(*nomes_bar)
df_bar_silver = df_bar_silver.filter(F.col("ano").rlike("^[0-9]{4}$"))

colunas_texto_bar = ["campo", "bacia", "estado", "situacao"]
df_bar_silver = corrigir_mojibake_colunas(df_bar_silver, colunas_texto_bar)
df_bar_silver = padronizar_texto(df_bar_silver, colunas_texto_bar)

# colunas numéricas "normais" -> formato americano
colunas_numericas_bar = ["voip_bbl", "vgip_m3", "petroleo_acumulado_bbl", "gas_natural_acumulado_m3"]
df_bar_silver = numero_us_para_double(df_bar_silver, colunas_numericas_bar)

# fração recuperada precisa remover o "%" antes de converter
valor_fracao = F.regexp_replace(F.col("fracao_recuperada_petroleo").cast("string"), "%", "")
df_bar_silver = df_bar_silver.withColumn(
    "fracao_recuperada_petroleo",
    (valor_fracao.try_cast(DoubleType()) / 100)  # guarda como fração (0.0379), não como percentual (3.79)
)

df_bar_silver = df_bar_silver.withColumn("ano", F.col("ano").cast("int"))
df_bar_silver.printSchema()
display(df_bar_silver.limit(5))

In [0]:
nomes_bmp = [
    "ano", "mes_ano", "estado", "bacia", "campo", "poco", "ambiente", "instalacao",
    "producao_oleo_m3", "producao_condensado_m3", "producao_gas_associado_mm3",
    "producao_gas_nao_associado_mm3", "producao_agua_m3", "injecao_gas_mm3",
    "injecao_agua_recuperacao_secundaria_m3", "injecao_agua_descarte_m3",
    "injecao_gas_carbonico_mm3", "injecao_nitrogenio_mm3", "injecao_vapor_agua_t",
    "injecao_polimeros_m3", "injecao_outros_fluidos_m3",
]
df_bmp_silver = df_bmp.toDF(*nomes_bmp)
df_bmp_silver = padronizar_texto(df_bmp_silver, ["estado", "bacia", "campo", "poco", "ambiente", "instalacao"])

colunas_numericas_bmp = [c for c in df_bmp_silver.columns if c.startswith("producao_") or c.startswith("injecao_")]
df_bmp_silver = numero_br_para_double(df_bmp_silver, colunas_numericas_bmp)

df_bmp_silver.printSchema()
display(df_bmp_silver.limit(5))

In [0]:
# Antes de gravar, isole e conte as linhas malformadas
linhas_invalidas = df_bmp_silver.filter(~F.col("ano").rlike("^(19|20)[0-9]{2}$"))
qtd_invalidas = linhas_invalidas.count()
print(f"Linhas com 'ano' malformado: {qtd_invalidas} de {df_bmp_silver.count()}")

linhas_invalidas.select("ano", "mes_ano", "estado").show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

df_bmp_raw_com_arquivo = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/PUC_Sprint_2/anp/raw/bmp/*.csv")
    .withColumn("arquivo_origem", F.col("_metadata.file_path")))

primeira_coluna = df_bmp_raw_com_arquivo.columns[0]  # a coluna "ano" original

diagnostico_por_arquivo = (df_bmp_raw_com_arquivo
    .withColumn("malformado", ~F.col(primeira_coluna).rlike("^(19|20)[0-9]{2}$"))
    .groupBy("arquivo_origem")
    .agg(
        F.count("*").alias("total_linhas"),
        F.sum(F.col("malformado").cast("int")).alias("linhas_malformadas")
    )
    .withColumn("pct_malformado", F.round(F.col("linhas_malformadas") / F.col("total_linhas") * 100, 1))
    .orderBy(F.desc("linhas_malformadas")))

diagnostico_por_arquivo.show(50, truncate=False)

In [0]:
arquivos_problematicos = [row["arquivo_origem"] for row in
    diagnostico_por_arquivo.filter(F.col("pct_malformado") > 5).collect()]

print(f"{len(arquivos_problematicos)} arquivos serão relidos com multiLine=True")

df_bmp_corrigido = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(arquivos_problematicos))

# Confirma se o multiLine realmente resolveu
primeira_coluna_corrigida = df_bmp_corrigido.columns[0]
malformadas_depois = df_bmp_corrigido.filter(~F.col(primeira_coluna_corrigida).rlike("^(19|20)[0-9]{2}$")).count()
print(f"Malformadas após releitura com multiLine: {malformadas_depois} de {df_bmp_corrigido.count()}")

In [0]:

linhas_exemplo = (spark.read.text(arquivos_problematicos[0])  # producao-terra-2018-1trim.csv
    .filter(F.col("value").contains("ANAMB"))
    .limit(3)
    .collect())

for row in linhas_exemplo:
    print(repr(row["value"]))

In [0]:
df_bdep_silver = df_bdep.toDF(*[limpar_nome_generico(c) for c in df_bdep.columns])
df_bdep_silver.printSchema()
display(df_bdep_silver.limit(5))

In [0]:
df_cambio_silver = df_cambio.withColumnRenamed("valor", "cambio_usd_brl")
df_brent_silver = (df_brent
    .toDF(*[limpar_nome_generico(c) for c in df_brent.columns])
    .withColumnRenamed("period", "data")
    .withColumnRenamed("value", "preco_brent_usd"))

display(df_brent_silver.limit(5))
display(df_cambio_silver.limit(5))


In [0]:
for c in colunas_numericas_bar:
    nulos = df_bar_silver.filter(F.col(c).isNull()).count()
    print(f"BAR.{c}: {nulos} nulos")

for c in colunas_numericas_bmp:
    nulos = df_bmp_silver.filter(F.col(c).isNull()).count()
    print(f"BMP.{c}: {nulos} nulos")

In [0]:
# BMP: nulos vêm de blank na origem, ou de conversão que falhou?
nomes_originais_bmp = df_bmp.columns
mapa_bmp = dict(zip(nomes_bmp, nomes_originais_bmp))

print(f"Total de linhas BMP: {df_bmp_silver.count()}\n")
for c in colunas_numericas_bmp:
    original = mapa_bmp[c]
    vazio_origem = df_bmp.filter((F.col(original).isNull()) | (F.trim(F.col(original)) == "")).count()
    nulo_pos = df_bmp_silver.filter(F.col(c).isNull()).count()
    bug = nulo_pos - vazio_origem
    print(f"{c}: vazio na origem={vazio_origem} | nulo após conversão={nulo_pos} | possível bug={bug}")

In [0]:
# BAR: mesma lógica, considerando o filtro de rodapé já aplicado
nomes_originais_bar = df_bar.columns
mapa_bar = dict(zip(nomes_bar, nomes_originais_bar))
df_bar_valido = df_bar.filter(F.col(nomes_originais_bar[0]).rlike("^[0-9]{4}$"))

print(f"Total de linhas BAR: {df_bar_silver.count()}\n")
for c in colunas_numericas_bar:
    original = mapa_bar[c]
    vazio_origem = df_bar_valido.filter((F.col(original).isNull()) | (F.trim(F.col(original)) == "")).count()
    nulo_pos = df_bar_silver.filter(F.col(c).isNull()).count()
    bug = nulo_pos - vazio_origem
    print(f"{c}: vazio na origem={vazio_origem} | nulo após conversão={nulo_pos} | possível bug={bug}")

In [0]:
for c in colunas_numericas_bar:
    original = mapa_bar[c]
    diagnostico = (df_bar_valido
        .withColumn("valor_original", F.col(original).cast("string"))
        .withColumn("limpo", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("valor_original"), r'["\n\r]', ""), r"\.", ""), ",", "."))
        .withColumn("convertido", F.col("limpo").try_cast(DoubleType()))
        .filter(
            (F.col("valor_original").isNotNull()) &
            (F.trim(F.col("valor_original")) != "") &
            (F.col("convertido").isNull())
        )
    )
    print(f"--- {c} ---")
    diagnostico.select("valor_original").distinct().show(15, truncate=False)

In [0]:
original_fracao = mapa_bar["fracao_recuperada_petroleo"]

diagnostico_fracao = (df_bar_valido
    .withColumn("valor_original", F.col(original_fracao).cast("string"))
    .withColumn("limpo", F.regexp_replace(F.col("valor_original"), "%", ""))
    .withColumn("convertido", F.col("limpo").try_cast(DoubleType()))
    .filter(
        (F.col("valor_original").isNotNull()) &
        (F.trim(F.col("valor_original")) != "") &
        (F.col("convertido").isNull())
    )
)
diagnostico_fracao.select("valor_original").distinct().show(15, truncate=False)

In [0]:
original_descarte = mapa_bmp["injecao_agua_descarte_m3"]

diagnostico = (df_bmp
    .withColumn("valor_original", F.col(original_descarte).cast("string"))
    .withColumn("limpo", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.col("valor_original"), r'["\n\r]', ""), r"\.", ""), ",", "."))
    .withColumn("convertido", F.col("limpo").try_cast(DoubleType()))
    .filter(
        (F.col("valor_original").isNotNull()) &
        (F.trim(F.col("valor_original")) != "") &
        (F.col("convertido").isNull())
    )
)
diagnostico.select("valor_original").distinct().show(30, truncate=False)

In [0]:
original_descarte = mapa_bmp["injecao_agua_descarte_m3"]

vazio_origem_corrigido = df_bmp.filter(
    F.col(original_descarte).isNull() |
    (F.regexp_replace(F.col(original_descarte), r'[\s"]+', "") == "")
).count()

nulo_pos = df_bmp_silver.filter(F.col("injecao_agua_descarte_m3").isNull()).count()
print(f"vazio na origem (corrigido)={vazio_origem_corrigido} | nulo após conversão={nulo_pos} | diferença={nulo_pos - vazio_origem_corrigido}")